In [ ]:
# ── Cell 1: CPU Setup ────────────────────────────────────────────────────────
# No GPU required — LightGBM only

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm>=4.3.0', 'scikit-learn>=1.4.0',
    'pandas', 'numpy', 'matplotlib'
], check=True)

print(f'PROJECT_DIR = {PROJECT_DIR}')
print('CPU setup complete. No GPU required for this notebook.')

In [ ]:
# ── Cell 2: Load Feature Arrays, Verify Shapes ───────────────────────────────

import numpy as np

t1 = np.load(f'{PROJECT_DIR}/feature_cache/tier1_features.npy')
t2 = np.load(f'{PROJECT_DIR}/feature_cache/tier2_oof_preds.npy')
t3 = np.load(f'{PROJECT_DIR}/feature_cache/tier3_features.npy')
y  = np.load(f'{PROJECT_DIR}/feature_cache/labels.npy')

print('Feature array shapes:')
print(f'  Tier 1 (NLI + Emb + Coherence + Epistemic) : {t1.shape}')
print(f'  Tier 2 (FinBERT OOF + DeBERTa OOF)        : {t2.shape}')
print(f'  Tier 3 (MLM Perplexity)                    : {t3.shape}')
print(f'  Labels                                     : {y.shape}')

# Shape sanity checks
assert t1.shape[0] == t2.shape[0] == t3.shape[0] == y.shape[0], (
    'Sample count mismatch across feature arrays!'
)
assert not np.isnan(t1).any(), 'NaN in tier1_features'
assert not np.isnan(t2).any(), 'NaN in tier2_oof_preds'
assert not np.isnan(t3).any(), 'NaN in tier3_features'

# Combine
X = np.hstack([t1, t2, t3])
print(f'\nCombined feature matrix X : {X.shape}  (~96 dims)')
print(f'Label distribution        : {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# ── Cell 3: Run Ablation ──────────────────────────────────────────────────────
# Evaluates all tier combinations to quantify contribution of each tier.

from models.meta_classifier import run_ablation

ablation_configs = {
    'T1 only':         np.hstack([t1]),
    'T2 only':         np.hstack([t2]),
    'T3 only':         np.hstack([t3]),
    'T1 + T2':         np.hstack([t1, t2]),
    'T1 + T3':         np.hstack([t1, t3]),
    'T2 + T3':         np.hstack([t2, t3]),
    'T1 + T2 + T3':    np.hstack([t1, t2, t3]),
}

print('Running ablation across tier combinations (5-fold CV each) ...')
ablation_results = run_ablation(
    configs=ablation_configs,
    y=y,
    n_splits=5,
    random_state=42,
)

import pandas as pd
df_ablation = pd.DataFrame(ablation_results)
df_ablation = df_ablation.sort_values('accuracy', ascending=False)
print('\nAblation Results:')
print(df_ablation.to_string(index=False))

df_ablation.to_csv(f'{PROJECT_DIR}/results/ablation_results.csv', index=False)
print(f'\nSaved: {PROJECT_DIR}/results/ablation_results.csv')

In [ ]:
# ── Cell 4: Train Final Model, Save meta_model.pkl ───────────────────────────
# Train on full dataset with best hyperparameters (no held-out test here —
# evaluation is on the organizer's blind set).

import pickle
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

lgb_params = dict(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

# Final nested 5-fold CV evaluation on combined feature matrix
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_meta = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    clf = lgb.LGBMClassifier(**lgb_params)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )
    oof_preds_meta[val_idx] = clf.predict(X_val)
    fold_acc = accuracy_score(y_val, clf.predict(X_val))
    print(f'  Fold {fold+1}/5 — val accuracy: {fold_acc:.4f}')

# Overall OOF metrics
acc  = accuracy_score(y, oof_preds_meta)
prec = precision_score(y, oof_preds_meta)
rec  = recall_score(y, oof_preds_meta)
f1   = f1_score(y, oof_preds_meta)
print(f'\nFull OOF — Acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}')

# Train final model on all data
final_model = lgb.LGBMClassifier(**lgb_params)
final_model.fit(X, y)

model_path = f'{PROJECT_DIR}/models/meta_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print(f'\n✅ Final meta-model saved: {model_path}')

In [ ]:
# ── Cell 5: Feature Importance Plot ──────────────────────────────────────────

import pickle
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb

with open(f'{PROJECT_DIR}/models/meta_model.pkl', 'rb') as f:
    final_model = pickle.load(f)

# Build feature name list matching the column order in X
# Tier 1: NLI(5) + Embeddings(varies) + Coherence(4) + Epistemic(3)
# Tier 2: FinBERT(2) + DeBERTa(2)
# Tier 3: Perplexity(4)
nli_names      = ['nli_contra_ratio', 'nli_max_contra', 'nli_entail_ratio',
                  'nli_coherence', 'nli_weighted_contra']
emb_names      = [f'emb_pca_{i}' for i in range(t1.shape[1] - 5 - 4 - 3)] + \
                  ['emb_cosine_dist', 'emb_mahal_dist']
coherence_names = ['coh_min_sim', 'coh_std_sim', 'coh_first_last', 'coh_mean']
epistemic_names = ['epi_cert_ratio', 'epi_hedge_density', 'epi_mismatch']
t2_names       = ['fin_p_false', 'fin_p_true', 'deb_p_false', 'deb_p_true']
t3_names       = ['ppl_mean', 'ppl_max', 'ppl_std', 'ppl_top10_ratio']

feature_names = nli_names + emb_names + coherence_names + epistemic_names + t2_names + t3_names

# Trim/pad to match actual feature count
n_features = X.shape[1]
if len(feature_names) != n_features:
    feature_names = [f'feat_{i}' for i in range(n_features)]
    print(f'Note: using generic feature names (expected {n_features})')

# Plot top-30 features by gain
fig, ax = plt.subplots(figsize=(10, 12))
lgb.plot_importance(
    final_model,
    ax=ax,
    importance_type='gain',
    max_num_features=30,
    title='LightGBM Feature Importance (Gain) — Top 30',
)

# Attempt to label with human-readable names
try:
    importances = final_model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:30]
    top_names = [feature_names[i] for i in top_idx]
    ax.set_yticklabels(top_names[::-1], fontsize=9)
except Exception:
    pass

plt.tight_layout()
plot_path = f'{PROJECT_DIR}/results/feature_importance.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Feature importance plot saved: {plot_path}')